In [0]:
import numpy as np
import pandas as pd


In [0]:
df= spark.table("default.titanic_clean").toPandas()

In [0]:
df["PassengerId"].duplicated().sum()

np.int64(0)

In [0]:
df["PassengerId"].nunique()

891

In [0]:
df = df.drop_duplicates(subset="PassengerId", keep = "first")

In [0]:
df.shape

(891, 12)

In [0]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

In [0]:
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

In [0]:
df[["SibSp","Parch","FamilySize", "IsAlone"]].head(10)

,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1
5,0,0,1,1
6,0,0,1,1
7,3,1,5,0
8,0,2,3,0
9,1,0,2,0


In [0]:
df["Name"].head(10)

0                              Braund, Mr. Owen Harris
1    Cumings, Mrs. John Bradley (Florence Briggs Th...
2                               Heikkinen, Miss. Laina
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                             Allen, Mr. William Henry
5                                     Moran, Mr. James
6                              McCarthy, Mr. Timothy J
7                       Palsson, Master. Gosta Leonard
8    Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)
9                  Nasser, Mrs. Nicholas (Adele Achem)
Name: Name, dtype: object

In [0]:
df["Title"] = df["Name"].str.extract(r',\s*([^.]*)\.')

In [0]:
df[["Name", "Title"]].head(10)

,Name,Title
0,"Braund, Mr. Owen Harris",Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,"Heikkinen, Miss. Laina",Miss
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs
4,"Allen, Mr. William Henry",Mr
5,"Moran, Mr. James",Mr
6,"McCarthy, Mr. Timothy J",Mr
7,"Palsson, Master. Gosta Leonard",Master
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",Mrs
9,"Nasser, Mrs. Nicholas (Adele Achem)",Mrs


In [0]:
df["Title"].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Mlle              2
Major             2
Col               2
the Countess      1
Capt              1
Ms                1
Sir               1
Lady              1
Mme               1
Don               1
Jonkheer          1
Name: count, dtype: int64

In [0]:
common_titles = ["Mr", "Miss", "Mrs", "Master"]

In [0]:
df["Title"] = df["Title"].where(
    df["Title"].isin(common_titles),"Other"
)

In [0]:
df["Age"].describe()

count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64

In [0]:
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins = [0,12,17,59,100],
    labels=  ["Child", "Teen", "Adult", "Senior"]
)

In [0]:
df[["Age", "AgeGroup"]].head()

,Age,AgeGroup
0,22.0,Adult
1,38.0,Adult
2,26.0,Adult
3,35.0,Adult
4,35.0,Adult


In [0]:
df["AgeGroup"].value_counts()

AgeGroup
Adult     575
Child      69
Teen       44
Senior     26
Name: count, dtype: int64

In [0]:
age_median = df["Age"].median()
age_median

np.float64(28.0)

In [0]:
df["Age"] = df["Age"].fillna(age_median)

In [0]:
df["AgeGroup"].value_counts()

AgeGroup
Adult     575
Child      69
Teen       44
Senior     26
Name: count, dtype: int64

In [0]:
df["FamilySizeGroup"] = pd.cut(
    df["FamilySize"],
    bins = [0,1,4,6, float("inf")],
    labels = ["Alone","Small","Medium","Large"]
)

In [0]:
df[["FamilySize", "FamilySizeGroup"]].head(10)

,FamilySize,FamilySizeGroup
0,2,Small
1,2,Small
2,1,Alone
3,2,Small
4,1,Alone
5,1,Alone
6,1,Alone
7,5,Medium
8,3,Small
9,2,Small


In [0]:
df["FamilySizeGroup"].value_counts()

FamilySizeGroup
Alone     537
Small     292
Medium     37
Large      25
Name: count, dtype: int64

In [0]:
df["Fare"].describe()

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64

In [0]:
df[["Sex", "Embarked", "Title", "AgeGroup", "FamilySizeGroup"]].head(10)

,Sex,Embarked,Title,AgeGroup,FamilySizeGroup
0,male,S,Mr,Adult,Small
1,female,C,Mrs,Adult,Small
2,female,S,Miss,Adult,Alone
3,female,S,Mrs,Adult,Small
4,male,S,Mr,Adult,Alone
5,male,Q,Mr,NaN,Alone
6,male,S,Mr,Adult,Alone
7,male,S,Master,Child,Medium
8,female,S,Mrs,Adult,Small
9,female,C,Mrs,Teen,Small


In [0]:
df.select_dtypes(include="object").columns

Index(['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked', 'Title'], dtype='object')

## 8. Save Processed Data

Persist the engineered, pre-encoding dataset for the next stage.

In [0]:
spark_df_processed = spark.createDataFrame(df)

spark.sql("DROP TABLE IF EXISTS default.titanic_processed")

spark_df_processed.write \
    .mode("overwrite") \
    .saveAsTable("default.titanic_processed")

## Conclusion

Feature engineering created family-size, isolation, title, age-group, family-size-group, and cabin-deck features. The processed dataset is saved as `default.titanic_processed` for feature encoding.